In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

import gplately

from lib.main import *
from lib.plot import *

from parameters import parameters

In [2]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 20

In [3]:
# muller22

rotation_model = [
    plate_model_dir+"/Rotations/1000_0_rotfile_MantleOpt.rot",
    plate_model_dir+"/Rotations/1000_0_rotfile_Merdith_etal_opt.rot",
    plate_model_dir+"/Rotations/lat_lon_velocity_domain_30_60.gpml",
    plate_model_dir+"/Rotations/no_net_rotation_model.rot",
]

topology_features = [
    plate_model_dir+"/Topologies/250-0_plate_bounds.gpml",
    plate_model_dir+"/Topologies/410-250_plate_bounds.gpml",
    plate_model_dir+"/Topologies/1000-410-Convergence.gpml",
    plate_model_dir+"/Topologies/1000-410-Divergence.gpml",
    plate_model_dir+"/Topologies/1000-410-Topologies.gpml",
    plate_model_dir+"/Topologies/1000-410-Transforms.gpml",
    plate_model_dir+"/Topologies/TopologyBuildingBlocks.gpml",
]

static_polygons = plate_model_dir+"/shapes_static_polygons_Merdith_etal.gpml"
coastlines = plate_model_dir+"/shapes_coastlines_Merdith_etal.gpmlz"
continents = plate_model_dir+"/shapes_continents.gpml"
COBs = plate_model_dir+"/COB_polygons_and_coastlines_combined_1000_0_Merdith_etal.gpml"

In [4]:
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        verbose=True,
    )
    
    subduction_data.to_csv(subduction_data_filename, index=False)

[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:   11.5s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:   12.7s
[Parallel(n_jobs=20)]: Done  21 tasks      | elapsed:   17.8s
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed:   20.3s
[Parallel(n_jobs=20)]: Done  45 tasks      | elapsed:   27.2s
[Parallel(n_jobs=20)]: Done  58 tasks      | elapsed:   28.1s
[Parallel(n_jobs=20)]: Done  73 tasks      | elapsed:   34.5s
[Parallel(n_jobs=20)]: Done  88 tasks      | elapsed:   40.8s
[Parallel(n_jobs=20)]: Done 105 tasks      | elapsed:   47.1s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:   52.3s
[Parallel(n_jobs=20)]: Done 141 tasks      | elapsed:   57.7s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:  1.1min
[Parallel(n_jobs=20)]: Done 181 tasks      | elapsed:  1.2min
[Parallel(n_jobs=20)]: Done 202 tasks      | elapsed:  1.3min
[Parallel(n_jobs=20)]: Done 225 tasks      | elapsed:  

In [5]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

projection = ccrs.Mollweide(central_longitude=60)

In [6]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    ax.set_global()

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)
    
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 50, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)

    # gplot.plot_all_topologies(ax, color="orangered", zorder=5)
    gplot.plot_ridges_and_transforms(ax, color="orangered", zorder=5)
    gplot.plot_trenches(ax, color="dimgray", zorder=6)
            
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='black', alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
    
    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(feat, shrink=0.4, pad=0.06, orientation="horizontal", extend="both")
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none", label="Continental Crust"),
        Line2D([0], [0], color="orangered", label="Mid-Ocean Ridges"),
        Line2D([0], [0], color="dimgray", label="Trench Lines")
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2))
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

In [7]:
@interact
def show_hist(feature=features_plot):
    # Create the plot
    fig, ax = plt.subplots()
    ax.set_facecolor('whitesmoke')
    counts_neg, bins, _ = ax.hist(subduction_data[feature], bins=50, facecolor='LightSalmon', edgecolor='black', density=True)
    ax.set_xlabel(format_feature_name(feature))
    ax.set_ylabel('Probability Density')
    fig.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='feature', options=('convergence_rate (cm/yr)', 'convergence_obliqu…

In [ ]:
if os.path.exists(feat_maps_dir):
    print(f"Feature maps are located in {feat_maps_dir}")
else:
    os.makedirs(feat_maps_dir, exist_ok=True)
       
    generate_feat_maps(
        rotation_model,
        topology_features,
        static_polygons,
        coastlines,
        continents,
        COBs,
        subduction_data,
        projection,
        time_steps,
        feature="convergence_rate (cm/yr)",
        output_dir=feat_maps_dir,
        n_jobs=nprocs
    )
    
    output_filenames = [
        os.path.join(feat_maps_dir, f"feat_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "feat_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )